In [ ]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import torch

In [ ]:
print("Loading dataset...")
dataset = load_dataset("stanfordnlp/imdb")



Loading dataset...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
train_split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

train_data = train_split["train"]
val_data = train_split["test"]
test_data = dataset["test"]

In [ ]:
X_train = train_data["text"]
y_train = train_data["label"]

X_val = val_data["text"]
y_val = val_data["label"]

X_test = test_data["text"]
y_test = test_data["label"]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_val_tfidf = vectorizer.transform(X_val)

X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

C_values = [0.001, 0.01, 0.1, 1, 10, 100]

best_C = None
best_val_acc = 0

for C in C_values:
    model = LogisticRegression(
        C=C,
        max_iter=1000
    )

    # Training on training set
    model.fit(X_train_tfidf, y_train)

    # Validation test
    val_pred = model.predict(X_val_tfidf)
    val_acc = accuracy_score(y_val, val_pred)

    print(f"C = {C:<6} | Validation Accuracy = {val_acc:.4f}")

    # finding best c
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_C = C

print("\nBest C:", best_C)
print("Best Validation Accuracy:", best_val_acc)

C = 0.001  | Validation Accuracy = 0.7812
C = 0.01   | Validation Accuracy = 0.8042
C = 0.1    | Validation Accuracy = 0.8618
C = 1      | Validation Accuracy = 0.8874
C = 10     | Validation Accuracy = 0.8854
C = 100    | Validation Accuracy = 0.8586

Best C: 1
Best Validation Accuracy: 0.8874


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    C=1,
    max_iter=1000
)

model.fit(
    X_train_tfidf,
    y_train
)

LogisticRegression(C=1, max_iter=1000)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

test_predictions = model.predict(X_test_tfidf)

print("Test Accuracy:",
      accuracy_score(y_test, test_predictions))

print("Test F1:",
      f1_score(y_test, test_predictions))

Test Accuracy: 0.88164
Test F1: 0.8817865846350526


Baseline model(Tf-idf + logistic regression) produced accuracy : 88.16%

#Distilbert fine tuning
now we fine tune distilbert according to our training data and see how it performs on classification task

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True
    )

In [ ]:
tokenized_train = train_data.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_data.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_data.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_train[0])

{'text': 'Stage adaptations often have a major fault. They often come out looking like a film camera was simply placed on the stage (Such as "Night Mother"). Sidney Lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. Still, it\'s nice to look at for what it is. The chemistry between Michael Caine and Christopher Reeve is quite brilliant. The dynamics of their relationship are surprising. Caine is fantastic as always, and Reeve gets one of his few chances to really act.<br /><br />I confess that I\'ve never seen Ira Levin\'s play, but I hear that Jay Presson Allen\'s adaptation is faithful. The script is incredibly convoluted, and keeps you guessing. "Deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10', 'label': 1, 'input_ids': [101, 2754, 17241, 2411, 2031, 1037, 2350, 6346, 1012, 2027, 2411, 2272, 2041, 2559, 2066, 1037, 2143, 4950, 2001,

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
#text" removed as we dont need it anymore(we have tokenized text)
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_val = tokenized_val.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

In [ ]:
#checking whether token_type_ids exists and removing it if it does as we dont need it for this task
if "token_type_ids" in tokenized_train.column_names:
    tokenized_train = tokenized_train.remove_columns("token_type_ids")
    tokenized_val = tokenized_val.remove_columns("token_type_ids")
    tokenized_test = tokenized_test.remove_columns("token_type_ids")

NameError: name 'tokenized_train' is not defined

In [ ]:
print(tokenized_train[0])

{'label': 1, 'input_ids': [101, 2754, 17241, 2411, 2031, 1037, 2350, 6346, 1012, 2027, 2411, 2272, 2041, 2559, 2066, 1037, 2143, 4950, 2001, 3432, 2872, 2006, 1996, 2754, 1006, 2107, 2004, 1000, 2305, 2388, 1000, 1007, 1012, 11430, 11320, 11368, 1005, 1055, 3257, 7906, 1996, 2143, 4142, 1010, 2029, 2003, 2926, 3697, 2144, 1996, 3861, 3253, 2032, 2053, 2613, 4119, 1012, 2145, 1010, 2009, 1005, 1055, 3835, 2000, 2298, 2012, 2005, 2054, 2009, 2003, 1012, 1996, 6370, 2090, 2745, 19881, 1998, 5696, 20726, 2003, 3243, 8235, 1012, 1996, 10949, 1997, 2037, 3276, 2024, 11341, 1012, 19881, 2003, 10392, 2004, 2467, 1010, 1998, 20726, 4152, 2028, 1997, 2010, 2261, 9592, 2000, 2428, 2552, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1045, 18766, 2008, 1045, 1005, 2310, 2196, 2464, 11209, 20206, 1005, 1055, 2377, 1010, 2021, 1045, 2963, 2008, 6108, 2811, 2239, 5297, 1005, 1055, 6789, 2003, 11633, 1012, 1996, 5896, 2003, 11757, 9530, 6767, 7630, 3064, 1010, 1998, 7906, 2017, 16986, 1012, 100

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tokenized_train,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    tokenized_val,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator
)

In [ ]:
#check what dataloader did
batch = next(iter(train_loader))

print(batch.keys())


KeysView({'input_ids': tensor([[  101,  2034,  2125,  ...,     0,     0,     0],
        [  101,  2023,  2003,  ...,     0,     0,     0],
        [  101,  1045, 21090,  ...,     0,     0,     0],
        ...,
        [  101,  2023,  2143,  ...,     0,     0,     0],
        [  101,  2296,  2320,  ...,  1028,  1026,   102],
        [  101,  2045,  2003,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([0, 1, 1, 0, 1, 1, 0, 0])})


In [ ]:
#so 1 batch has 8 examples and sequence length is 512.
print(batch["input_ids"].shape)

torch.Size([8, 512])


In [ ]:
#Adam optimizer to update weights as the model trains
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Using:", device)

Using: cuda


In [ ]:
epochs = 1

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Training Loss: {avg_loss:.4f}"
    )

Epoch 1/1 | Training Loss: 0.2601


In [ ]:
#1 epoch isn't sufficient so same loop again.
epochs = 1

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Additional epoch | Training Loss: {avg_loss:.4f}")

Additional epoch | Training Loss: 0.1386


In [ ]:

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in val_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == batch["labels"]
        ).sum().item()

        total += batch["labels"].size(0)

val_accuracy = correct / total

print(f"Validation Accuracy: {val_accuracy:.4f}")

Validation Accuracy: 0.9124


In [ ]:
test_loader = DataLoader(
    tokenized_test,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator
)

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == batch["labels"]
        ).sum().item()

        total += batch["labels"].size(0)

test_accuracy = correct / total

print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.9188


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        labels = batch["labels"]

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.9188
Precision: 0.9461
Recall   : 0.8883
F1 Score : 0.9163


So distilbert classification model produced 91.88% accuracy which makes it better than the baseline model...